# generate data

In [12]:
import pandas as pd
import random
from faker import Faker
import calendar
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

def get_working_days(year, month):
    """Calculate the number of working days in a given month (excluding weekends)."""
    total_days = calendar.monthrange(year, month)[1]
    weekdays = sum(1 for day in range(1, total_days + 1) if calendar.weekday(year, month, day) < 5)
    return weekdays

def generate_fake_burnout_data(years=20):
    """Generate burnout data for 20 employees over the past 'years' years."""
    data = []
    daily_work_hours = 8  # Standard work hours per day
    employees = list(range(1, 41))  # Employee IDs from 1 to 20
    
    current_year = datetime.today().year
    start_year = current_year - years  # Start from 20 years ago

    for year in range(start_year, current_year + 1):
        for month in range(1, 13):  # Loop through all months
            month_str = f"{year}-{month:02d}"
            for emp_id in employees:
                working_days_per_month = get_working_days(year, month)
                num_leaves_taken = random.randint(0, min(5, working_days_per_month))  # Ensure valid leave count
                actual_working_days = max(working_days_per_month - num_leaves_taken, 0)
                total_work_hours = actual_working_days * daily_work_hours
                num_high_priority_tasks = random.randint(5, 40)
                overtime_frequency = random.randint(0, 25)
                task_backlog = random.randint(0, 20)

                # Define burnout conditions
                burnout_factors = sum([
                    total_work_hours > 200,
                    num_high_priority_tasks > 10,
                    num_leaves_taken > 3,
                    overtime_frequency > 10,
                    task_backlog > 10
                ])
                burnout = 1 if burnout_factors >= 3 else 0

                data.append([
                    emp_id, month_str, total_work_hours, num_high_priority_tasks, num_leaves_taken,
                    overtime_frequency, task_backlog, burnout
                ])

    columns = ["emp_id", "month", "total_work_hours", "num_high_priority_tasks", "num_leaves_taken", 
               "overtime_frequency", "task_backlog", "burnout"]
    df = pd.DataFrame(data, columns=columns)
    return df

# Generate fake burnout data for the past 20 years
df_fake_burnout = generate_fake_burnout_data(years=20)

# Save to CSV
df_fake_burnout.to_csv("burnout_training_data_20_years.csv", index=False)

# Preview the first 5 rows
df_fake_burnout.head()


,emp_id,month,total_work_hours,num_high_priority_tasks,num_leaves_taken,overtime_frequency,task_backlog,burnout
0,1,2005-01,128,12,5,0,8,0
1,2,2005-01,160,19,1,4,3,0
2,3,2005-01,128,39,5,2,18,1
3,4,2005-01,144,7,3,0,2,0
4,5,2005-01,160,19,1,16,19,1


In [96]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define years and employees
years = list(range(2005, 2025))  # 20 years
months = [f"{m:02d}" for m in range(1, 13)]  # "01" to "12"
emp_ids = list(range(1, 51))  # Employees 1-20

data = []

for year in years:
    for month in months:
        for emp_id in emp_ids:
            # Generate realistic work hours
            total_work_hours = np.random.randint(110, 250)
            num_high_priority_tasks = np.random.randint(5, 40)
            num_leaves_taken = np.random.randint(0, 6)
            overtime_frequency = np.random.randint(0, 20)
            task_backlog = np.random.randint(0, 25)
            
            # Define burnout conditions (with noise)
            burnout_factors = sum([
                total_work_hours > 220,
                num_high_priority_tasks > 25,
                num_leaves_taken < 2,  
                overtime_frequency > 12,
                task_backlog > 15
            ])
            
            # Apply some randomness to burnout labeling (5% noise)
            burnout = 1 if burnout_factors >= 3 else 0
            if np.random.rand() < 0.05:  # Flip 5% of labels for noise
                burnout = 1 - burnout  
            
            # Append data
            data.append([emp_id, f"{year}-{month}", total_work_hours, num_high_priority_tasks, num_leaves_taken, overtime_frequency, task_backlog, burnout])

# Create DataFrame
df = pd.DataFrame(data, columns=["emp_id", "month", "total_work_hours", "num_high_priority_tasks", "num_leaves_taken", "overtime_frequency", "task_backlog", "burnout"])

# Save to CSV
df.to_csv("C:/Users/Sarah/Desktop/try1/burnout_data_20_years.csv", index=False)

print("✅ Data Generated & Saved!")


✅ Data Generated & Saved!


# train data

In [ ]:
"""
# Define burnout conditions
burnout_factors = sum([
    total_work_hours > 200,
    num_high_priority_tasks > 10,
    num_leaves_taken > 3,
    overtime_frequency > 10,
    task_backlog > 10
])
burnout = 1 if burnout_factors >= 2 else 0

"""

"C:\Users\Sarah\Desktop\try1\burnout_training_data_20_years.csv"
	emp_id	month	total_work_hours	num_high_priority_tasks	num_leaves_taken	overtime_frequency	task_backlog	burnout

In [97]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report


In [108]:
# data
file_path = r"C:\Users\Sarah\Desktop\try1\csv\burnout_data_20_years.csv"
df = pd.read_csv(file_path)

df.head()


,emp_id,month,total_work_hours,num_high_priority_tasks,num_leaves_taken,overtime_frequency,task_backlog,burnout
0,1,2005-01,212,33,2,7,20,0
1,2,2005-01,184,15,4,3,7,0
2,3,2005-01,162,6,3,5,1,0
3,4,2005-01,130,37,3,11,24,0
4,5,2005-01,168,32,3,15,14,0


In [109]:
# 2️⃣ Preprocessing
# Drop unnecessary columns (e.g., 'month' since it's not numerical)
df.drop(columns=['month'])


,emp_id,total_work_hours,num_high_priority_tasks,num_leaves_taken,overtime_frequency,task_backlog,burnout
0,1,212,33,2,7,20,0
1,2,184,15,4,3,7,0
2,3,162,6,3,5,1,0
3,4,130,37,3,11,24,0
4,5,168,32,3,15,14,0
...,...,...,...,...,...,...,...
11995,46,119,23,2,0,4,0
11996,47,218,6,3,13,5,0
11997,48,152,34,3,5,1,0
11998,49,173,19,0,9,8,0


In [110]:
# Features (X) and Target (y)
X = df.drop(columns=['burnout','month'])  # independent var (features)
y = df['burnout']  # dependent var (target)


In [111]:
# Train-test split (85% train, 15% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)


In [112]:
# standardize features (x) (Only for numeric values)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [113]:
import joblib

# Save the scaler
joblib.dump(scaler, r"C:\Users\Sarah\Desktop\try1\model\scaler.pkl")
print("Scaler saved successfully!")


Scaler saved successfully!


In [114]:
# Convert back to DataFrame to keep feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


In [115]:
# train
rf_model = RandomForestClassifier(
    n_estimators=100,        # Reduce tree count
    max_depth=5,             # Limit depth of trees
    min_samples_split=20,    # Require at least 10 samples per split
    min_samples_leaf=10,      # Require at least 4 samples per leaf
    max_features='sqrt',     # Use only sqrt(features) at each split
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
print(rf_model)

RandomForestClassifier(max_depth=5, min_samples_leaf=10, min_samples_split=20,
                       random_state=42)


In [116]:
# 🔹 Logistic Regression
log_reg = LogisticRegression(
    C=2.0,                 # Reduce regularization for better fit
    max_iter=2000,         # Allow more iterations for convergence
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)
log_reg.fit(X_train_scaled, y_train)

log_pred = log_reg.predict(X_test_scaled)
print(log_reg)


LogisticRegression(C=2.0, class_weight='balanced', max_iter=2000,
                   random_state=42)


In [117]:
# 🔹 XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,        # Reduce tree count
    learning_rate=0.1,       # Slower learning
    max_depth=4,             # Smaller trees
    min_child_weight=10,      # Require larger groups
    subsample=0.8,           # Use only 80% of data per tree
    colsample_bytree=0.8,    # Use 80% of features per tree
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train_scaled, y_train)

xgb_pred = xgb_model.predict(X_test_scaled)
print(xgb_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)


In [118]:
# 4️⃣ Model Evaluation
def evaluate_model(model_name, y_true, y_pred):
    print(f"\n🔹 Model: {model_name}")
    print(f"Accuracy Score: {accuracy_score(y_true, y_pred):.4f}")
    print("Classification Report:\n", classification_report(y_true, y_pred))

evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("Logistic Regression", y_test, log_pred)
evaluate_model("XGBoost", y_test, xgb_pred)



🔹 Model: Random Forest
Accuracy Score: 0.9461
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97      1379
           1       0.95      0.82      0.88       421

    accuracy                           0.95      1800
   macro avg       0.95      0.90      0.92      1800
weighted avg       0.95      0.95      0.94      1800


🔹 Model: Logistic Regression
Accuracy Score: 0.7833
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.78      0.85      1379
           1       0.52      0.79      0.63       421

    accuracy                           0.78      1800
   macro avg       0.72      0.79      0.74      1800
weighted avg       0.83      0.78      0.80      1800


🔹 Model: XGBoost
Accuracy Score: 0.9461
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97      1379
           1       0.95 

In [119]:
# 5️⃣ Cross-Validation Scores
def cross_validate(model, name):
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
    print(f"\n🔹 {name} Cross-Validation Accuracy: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

cross_validate(rf_model, "Random Forest")
cross_validate(log_reg, "Logistic Regression")
cross_validate(xgb_model, "XGBoost")


🔹 Random Forest Cross-Validation Accuracy: 0.9492 ± 0.0027

🔹 Logistic Regression Cross-Validation Accuracy: 0.7825 ± 0.0069

🔹 XGBoost Cross-Validation Accuracy: 0.9513 ± 0.0015


In [120]:
# 6️⃣ Show some predictions (First 10)
sample_predictions = pd.DataFrame({
    "Actual": y_test[:10].values,
    "Random Forest": rf_pred[:10],
    "Logistic Regression": log_pred[:10],
    "XGBoost": xgb_pred[:10]
})
print("\n🔍 Sample Predictions:")
print(sample_predictions)


🔍 Sample Predictions:
   Actual  Random Forest  Logistic Regression  XGBoost
0       1              1                    1        1
1       0              0                    0        0
2       0              0                    0        0
3       0              0                    0        0
4       1              1                    1        1
5       0              0                    0        0
6       0              0                    0        0
7       0              0                    0        0
8       0              0                    0        0
9       1              1                    1        1


In [ ]:
import joblib

# Save the trained model
joblib.dump(model, r"C:\Users\Sarah\Desktop\trend_analysis\model\random_forest_model.pkl")

print("Model saved successfully!")


In [ ]:
# Load the model
loaded_model = joblib.load(r"C:\Users\Sarah\Desktop\trend_analysis\model\random_forest_model.pkl")

print("Model loaded successfully!")

# Now you can make predictions using the loaded model
y_pred = loaded_model.predict(X_test)
